# Consolidated plant-controller iLQR

This notebook combines the recurrence, orthogonal, and overlapping implementations behind one configurable plant-controller model and one shared iLQR solver.

## Setup

Run this cell first. The notebook requires NumPy and PyTorch. Results are written beneath the notebook's current working directory.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import os

import numpy as np
import torch
from torch.autograd.functional import jacobian

torch.set_default_dtype(torch.float64)

## Controller, angle, and network selection

The allowed controller types are `recurrence`, `orthogonal`, and `overlapping`. Set `controller_type` to one of these values. Set `angle_selection` to `pi/8` (index 1), `pi/4` (index 2), or `3pi/8` (index 3). Set `w_type` to `non_normal` for $W=[[0,0],[1,0]]$ or `oscillatory` for $W=[[0,-1],[1,0]]$. Set `convergence_threshold` to the desired positive terminal-error threshold.

In [2]:
controller_types = ("recurrence", "orthogonal", "overlapping")
controller_type = "recurrence"
angle_options = {"pi/8": (1, 22.5), "pi/4": (2, 45.0), "3pi/8": (3, 67.5)}
angle_selection = "pi/8"
w_types = ("non_normal", "oscillatory")
w_type = "oscillatory"
convergence_threshold = 0.5
w_options = {
    "non_normal": ((0.0, 0.0), (1.0, 0.0)),
    "oscillatory": ((0.0, -1.0), (1.0, 0.0)),
}


## Optimization configuration

The production defaults retain the original horizon and preparatory period. iLQR is capped at 500 iterations. Convergence is defined as $|y-y^*|<$ `convergence_threshold`, while progress reports show the signed error $y-y^*$ every tenth iteration. Saved filenames use the index associated with the selected angle.

In [3]:
horizon = int(os.getenv("ILQR_HORIZON", "1750"))
prep_steps = int(os.getenv("ILQR_PREP_STEPS", "750"))
max_iterations = min(int(os.getenv("ILQR_MAX_ITERATIONS", "500")), 500)
save_every = 10
step_size = 0.1
output_root = Path(os.getenv("ILQR_OUTPUT_ROOT", "."))

if controller_type not in controller_types:
    raise ValueError("controller_type must be recurrence, orthogonal, or overlapping")
if angle_selection not in angle_options:
    raise ValueError("angle_selection must be pi/8, pi/4, or 3pi/8")
if w_type not in w_types:
    raise ValueError("w_type must be non_normal or oscillatory")
if not np.isfinite(convergence_threshold) or convergence_threshold <= 0:
    raise ValueError("convergence_threshold must be a positive finite number")
if horizon < 2:
    raise ValueError("horizon must be at least 2")
if not 0 <= prep_steps < horizon:
    raise ValueError("prep_steps must be between 0 and horizon - 1")

angle_index, angle_degrees = angle_options[angle_selection]
theta_c = float(np.deg2rad(angle_degrees))

## Consolidated plant-controller model

The state layout is `[x1, x2, y, y_dot, u1, u2, go_cue, goal1, goal2, goal3, goal4, goal5, trailing_u1, trailing_u2]`. The model variants preserve the original differences in the source of the plant input, the third goal, and the goal-cost weights.

| Goal | User-friendly purpose |
| --- | --- |
| `goal1` | Move `y` toward the desired target after the go cue. Its influence increases gradually across the trial. |
| `goal2` | During preparation, keep `y` near its initial position and keep its velocity near zero. |
| `goal3` | For `recurrence` and `orthogonal`, keep the second neural-state derivative small during preparation, helping separate preparatory and movement dynamics. For `overlapping`, keep the two neural-state derivatives similar, encouraging shared or overlapping dynamics. |
| `goal4` | After the go cue, keep the first neural-state derivative small. This goal is active for `recurrence` and `orthogonal`; its cost weight is zero for `overlapping`. |
| `goal5` | Measure the squared rate of change of the plant input, which can be used to discourage abrupt control changes. Its cost weight is currently zero for all controllers. |

The `goal_weights` tuple in each plant configuration lists the cost weights for `goal1` through `goal5` in that order.

In [4]:
@dataclass(frozen=True)
class PlantConfig:
    name: str
    control_source: str
    goal3_mode: str
    goal_weights: tuple
    output_directory: str
    file_label: str


PLANT_CONFIGS = {
    "recurrence": PlantConfig("recurrence", "neural", "orthogonal", (2.0, 1.0, 1.0, 1.0, 0.0), "data_recurrence", "recurrence"),
    "orthogonal": PlantConfig("orthogonal", "kinematic", "orthogonal", (2.0, 1.0, 1.0, 1.0, 0.0), "data_orthogonal", "orthogonal"),
    "overlapping": PlantConfig("overlapping", "kinematic", "overlapping", (1.0, 1.0, 0.1, 0.0, 0.0), "data_overlapping", "overlapping"),
}


class PlantController:
    state_size = 14
    control_size = 4

    def __init__(self, config, theta_c, angle_index, w_type, w_values, horizon, prep_steps, y_init=0.1, y_target=10.0, dt=0.001, tau=0.150):
        self.config = config
        self.theta_c = theta_c
        self.angle_index = angle_index
        self.w_type = w_type
        self.w_values = w_values
        self.horizon = horizon
        self.prep_steps = prep_steps
        self.y_init = y_init
        self.y_target = y_target
        self.dt = dt
        self.tau = tau

    def initial_state(self):
        state = np.zeros(self.state_size, dtype=np.float64)
        state[2:4] = self.y_init
        state += 1e-16
        return state

    def desired_state(self):
        state = np.zeros(self.state_size, dtype=np.float64)
        state[2] = self.y_target
        state[6] = 1.0
        return state

    def __call__(self, x_k, u_k, step_index):
        dtype = x_k.dtype
        device = x_k.device
        W = torch.tensor(self.w_values, dtype=dtype, device=device)
        C_i = torch.tensor((np.cos(self.theta_c), np.sin(self.theta_c)), dtype=dtype, device=device)
        source = x_k[:2] if self.config.control_source == "neural" else x_k[2:4]
        u_plant_dot = u_k.reshape(2, 2) @ source
        u_plant_k1 = x_k[4:6] + self.dt * u_plant_dot
        y_dot_dot = C_i @ x_k[:2]
        y_dot_k1 = x_k[[3]] + self.dt * y_dot_dot
        y_k1 = x_k[[2]] + self.dt * x_k[[3]]
        x_ss_dot = (-x_k[:2] + W @ torch.tanh(x_k[:2]) + x_k[4:6] - 10.0) / self.tau
        x_ss_k1 = x_k[:2] + self.dt * x_ss_dot
        go_cue = torch.tensor((0.0 if step_index < self.prep_steps else 1.0,), dtype=dtype, device=device)
        goal1 = (step_index / self.horizon) * x_k[[6]] * (x_k[[2]] - self.y_target)
        goal2 = (1.0 - x_k[[6]]) * ((x_k[[2]] - self.y_init) + x_k[[3]])
        if self.config.goal3_mode == "orthogonal":
            goal3 = (1.0 - x_k[[6]]) * x_ss_dot[[1]]
        else:
            goal3 = x_ss_dot[[0]] - x_ss_dot[[1]]
        goal4 = x_k[[6]] * x_ss_dot[[0]]
        goal5 = (u_plant_dot.square().sum()).reshape(1)
        return torch.cat((x_ss_k1, y_k1, y_dot_k1, u_plant_k1, go_cue, goal1, goal2, goal3, goal4, goal5, u_plant_k1))

## Shared iLQR implementation

This solver is used unchanged for all three models. After each update it evaluates the new trajectory, stops when the absolute terminal error is below the user-specified `convergence_threshold`, and never exceeds 500 iterations. It saves the four output trajectories every tenth iteration and again when convergence is reached. Every tenth iteration it prints terminal `y` and the signed `error = y - y*`.

In [5]:
class ILQRSolver:
    def __init__(self, plant, step_size=0.1, max_iterations=500, convergence_threshold=0.5, save_every=10, output_root=Path(".")):
        self.plant = plant
        self.step_size = step_size
        self.max_iterations = min(max_iterations, 500)
        self.convergence_threshold = convergence_threshold
        self.save_every = save_every
        self.output_root = Path(output_root)
        self.Q, self.Qf, self.R = self._cost_matrices()

    def _cost_matrices(self):
        Q = np.zeros((self.plant.state_size, self.plant.state_size), dtype=np.float64)
        for index, weight in enumerate(self.plant.config.goal_weights, start=7):
            Q[index, index] = weight
        return Q, Q.copy(), np.eye(self.plant.control_size, dtype=np.float64) * 5e-7

    def rollout(self, controls):
        states = [self.plant.initial_state()]
        state = torch.as_tensor(states[0])
        for step_index, control in enumerate(controls):
            state = self.plant(state, torch.as_tensor(control), step_index)
            states.append(state.detach().numpy())
        return np.asarray(states)

    def linearize(self, states, controls):
        A_values = []
        B_values = []
        for step_index, control in enumerate(controls):
            state_t = torch.as_tensor(states[step_index])
            control_t = torch.as_tensor(control)
            dynamics = lambda state, action: self.plant(state, action, step_index)
            A_t, B_t = jacobian(dynamics, (state_t, control_t))
            A_values.append(A_t.detach().numpy())
            B_values.append(B_t.detach().numpy())
        return np.asarray(A_values), np.asarray(B_values)

    def backward_pass(self, states, controls, A_values, B_values):
        S_next = self.Qf.copy()
        v_next = self.Qf @ (states[-1] - self.plant.desired_state()).reshape(-1, 1)
        gains = [None] * len(controls)
        value_gains = [None] * len(controls)
        control_gains = [None] * len(controls)
        values = [None] * (len(controls) + 1)
        values[-1] = v_next
        for step_index in range(len(controls) - 1, -1, -1):
            A_t = A_values[step_index]
            B_t = B_values[step_index]
            system = B_t.T @ S_next @ B_t + self.R
            K_t = np.linalg.solve(system, B_t.T @ S_next @ A_t)
            Kv_t = np.linalg.solve(system, B_t.T)
            Ku_t = np.linalg.solve(system, self.R)
            closed_loop = A_t - B_t @ K_t
            S_t = A_t.T @ S_next @ closed_loop + self.Q
            v_t = closed_loop.T @ v_next - K_t.T @ self.R @ controls[step_index].reshape(-1, 1) + self.Q @ states[step_index].reshape(-1, 1)
            gains[step_index] = K_t
            value_gains[step_index] = Kv_t
            control_gains[step_index] = Ku_t
            values[step_index] = v_t
            S_next = S_t
            v_next = v_t
        return np.asarray(gains), np.asarray(value_gains), np.asarray(control_gains), np.asarray(values)

    def control_update(self, controls, A_values, B_values, gains, value_gains, control_gains, values):
        delta_state = np.zeros((self.plant.state_size, 1), dtype=np.float64)
        delta_controls = []
        for step_index in range(len(controls)):
            delta_control = -gains[step_index] @ delta_state - value_gains[step_index] @ values[step_index + 1] - control_gains[step_index] @ controls[step_index].reshape(-1, 1)
            delta_state = A_values[step_index] @ delta_state + B_values[step_index] @ delta_control
            delta_controls.append(delta_control.ravel())
        return np.asarray(delta_controls)

    def save_trajectories(self, states):
        directory = self.output_root / self.plant.config.output_directory / self.plant.w_type
        directory.mkdir(parents=True, exist_ok=True)
        label = self.plant.config.file_label
        np.save(directory / f"ss_{label}_{self.plant.angle_index}.npy", states[1:, :2])
        np.save(directory / f"y_{label}_{self.plant.angle_index}.npy", states[1:, 2:4])
        np.save(directory / f"gocue_{label}_{self.plant.angle_index}.npy", states[1:, 6:7])
        np.save(directory / f"uinput_traj_{label}_{self.plant.angle_index}.npy", states[1:, -2:])

    def solve(self):
        controls = np.zeros((self.plant.horizon - 1, self.plant.control_size), dtype=np.float64)
        history = []
        converged = False
        states = self.rollout(controls)
        for iteration in range(1, self.max_iterations + 1):
            A_values, B_values = self.linearize(states, controls)
            gains, value_gains, control_gains, values = self.backward_pass(states, controls, A_values, B_values)
            delta_controls = self.control_update(controls, A_values, B_values, gains, value_gains, control_gains, values)
            controls = controls + self.step_size * delta_controls
            states = self.rollout(controls)
            y_value = float(states[-1, 2])
            error = y_value - self.plant.y_target
            history.append((iteration, y_value, error, float(np.abs(delta_controls).sum())))
            if iteration % self.save_every == 0:
                self.save_trajectories(states)
                print(f"iteration={iteration:03d}  y={y_value:.6f}  error={error:.6f}")
            if abs(error) < self.convergence_threshold:
                converged = True
                if iteration % self.save_every != 0:
                    self.save_trajectories(states)
                    print(f"converged at iteration={iteration:03d}  y={y_value:.6f}  error={error:.6f}")
                break
        return {"controls": controls, "states": states, "history": np.asarray(history), "iterations": len(history), "converged": converged}


## Run the selected model

Run this cell to optimize the selected configuration. The returned `result` dictionary contains the optimized controls, state trajectory, iteration history, iteration count, and convergence flag. Saved arrays are produced every tenth iteration and on the iteration when convergence is reached.

In [ ]:
plant = PlantController(
    config=PLANT_CONFIGS[controller_type],
    theta_c=theta_c,
    angle_index=angle_index,
    w_type=w_type,
    w_values=w_options[w_type],
    horizon=horizon,
    prep_steps=prep_steps,
)

solver = ILQRSolver(
    plant=plant,
    step_size=step_size,
    max_iterations=max_iterations,
    convergence_threshold=convergence_threshold,
    save_every=save_every,
    output_root=output_root,
)

result = solver.solve()
print(f"finished: controller={controller_type}, w_type={w_type}, angle_index={angle_index}, angle={angle_degrees:g} degrees, iterations={result['iterations']}, converged={result['converged']}")

iteration=010  y=-2558.736567  error=-2568.736567


## Run another implementation

Change `controller_type`, `angle_selection`, or `w_type` in the selection cell, rerun the configuration cells, and then rerun the final cell. Each controller writes to its own output directory, with separate `non_normal` and `oscillatory` subfolders. Filenames end in `_1`, `_2`, or `_3` according to the selected angle.